# Mixed-layer depth (Stevens, 2010)

This notebook computes the mixed-layer depth (MLD) of the OAE ensemble from the ROMS history output (`temp`, `salt`, `zeta`). The ALK-fraction step in `OAE_ensemble.ipynb` uses this MLD as its upper-layer depth (`target_depths`).

Following Gerke et al. (2016) and Stevens (2010), the MLD is the shallowest depth at which the mean density has increased by half of the mean density difference between the surface and the depth of maximum buoyancy frequency:

$$
mld = \min\left\{ z : \overline{\sigma}_z - \overline{\sigma}_{surf} \ge 0.5 \cdot \left[\overline{\sigma}_{z_N} - \overline{\sigma}_{surf}\right] \right\}
$$

where $\overline{\sigma}_z$ is the time- and area-mean potential density at depth $z$, $\overline{\sigma}_{surf}$ is the mean surface density, and $\overline{\sigma}_{z_N}$ is the mean density at $z_N$, the depth at which the buoyancy frequency $N^2 = (g/\rho_0)\,\partial\overline{\sigma}/\partial z$ of the mean profile is at its maximum.

**Implementation**
- $\sigma_0$ is computed with TEOS-10 (`gsw`) from ROMS potential temperature and practical salinity.
- Depth $z$ is measured down from the instantaneous free surface. Each column's $\sigma_0$ is linearly interpolated to fixed depth levels (`mld_depth_levels`) before averaging. The surface value is the shallowest model level.
- The area mean is weighted by cell area $1/(p_m p_n)$ over ocean points (optionally within `mld_region`). At each depth, only columns deep enough to reach it are included. The time mean covers every history record, with records duplicated across restart files counted once.
- $z_N$ is taken at the mid-point between the two depth levels with the largest $N^2$, and the MLD is linearly interpolated between depth levels.
- The MLD is computed for each ensemble member, and for each season from the pooled July (CDR1–3) and November (CDR4–6) profiles.

**Output:** `Iceland3_mld_stevens2010.nc` in `out_dir`, with the mean $\sigma_0$ and $N^2$ profiles, $z_N$ and the MLD for every member and season.

In [ ]:
import numpy as np

# =========================
# Paths (Anvil)
# =========================
model_grid_path = "/anvil/projects/x-ees250129/x-uheede/Iceland_experiments/Iceland3_MARBL_2024/P_INPUT/Iceland3_grid.nc"
experiment_dir = "/anvil/projects/x-ees250129/x-uheede/Iceland_experiments"
out_dir = "/home/x-uheede/R/Intermediate_files/"

# Ensemble members by release season, and the month of their history files
seasons = {
    "July": ["CDR1", "CDR2", "CDR3"],
    "November": ["CDR4", "CDR5", "CDR6"],
}
his_months = {"July": "202407", "November": "202411"}

his_patterns = {
    ens: f"{experiment_dir}/Iceland3_MARBL_2024_{ens}/Iceland3_MARBL_2024_his.{his_months[season]}*.nc"
    for season, members in seasons.items()
    for ens in members
}

# =========================
# MLD settings
# =========================
mld_depth_levels = np.arange(0.0, 50.0 + 0.25, 0.25)  # m below the free surface
mld_region = None  # None = all ocean points, or (lon_min, lon_max, lat_min, lat_max) in degrees E / N
mld_time_stride = 1  # use every n-th history record (1 = all records)

In [ ]:
import glob
import os

import gsw
import numpy as np
import xarray as xr
from roms_tools import Grid
from roms_tools.vertical_coordinate import sigma_stretch

G = 9.81  # m s-2
RHO0 = 1025.0  # kg m-3, reference density in N^2 = (g / rho0) d(sigma)/dz


def roms_level_depths(zeta, h):
    """Depth of the rho levels below the free surface (m, positive down), top level first.

    UCLA-ROMS vertical coordinate (Vtransform=2) with the grid's s-coordinate
    (sc_r, cs_r, hc; set below from the grid file). zeta, h: (n_cols,).
    Returns (vert_levels, n_cols).
    """
    s = (hc * sc_r[:, None] + h * cs_r[:, None]) / (hc + h)  # z = zeta + (zeta + h) * s
    return (-(zeta + h) * s)[::-1]  # depth below the free surface = zeta - z


def interp_to_depths(values, depth, targets, block=20000):
    """Linearly interpolate every column of `values` to the depths `targets`.

    values, depth: (n_levels, n_cols), with depth increasing down each column.
    Values are held constant above the shallowest and below the deepest level.
    Returns (n_targets, n_cols).
    """
    n_levels, n_cols = depth.shape
    out = np.empty((targets.size, n_cols))
    for start in range(0, n_cols, block):
        d = depth[:, start : start + block]
        v = values[:, start : start + block]
        n = d.shape[1]
        # Shift each column by its own offset so that all columns form one sorted
        # array; a single searchsorted then counts the levels above each target.
        offset = (targets.max() + d.max() + 1.0) * np.arange(n)
        pos = np.searchsorted((d + offset).T.ravel(), (targets[:, None] + offset).ravel())
        k = np.clip(pos.reshape(targets.size, n) - n_levels * np.arange(n), 1, n_levels - 1)
        d_lo = np.take_along_axis(d, k - 1, axis=0)
        d_hi = np.take_along_axis(d, k, axis=0)
        v_lo = np.take_along_axis(v, k - 1, axis=0)
        v_hi = np.take_along_axis(v, k, axis=0)
        w = np.clip((targets[:, None] - d_lo) / (d_hi - d_lo), 0.0, 1.0)
        out[:, start : start + n] = v_lo + w * (v_hi - v_lo)
    return out


def accumulate_sigma_profile(his_glob, cols, h, area, saar, targets, time_stride=1):
    """Area-weighted sums of sigma0 on the depths `targets`, over all history records.

    Only the rho columns `cols` (flat indices) are used, and a depth counts only
    where it lies within the water column. Returns (sum of area * sigma0,
    sum of area, number of records); keeping the sums lets runs be pooled.
    """
    files = sorted(glob.glob(his_glob))
    if not files:
        raise FileNotFoundError(f"No history files match {his_glob}")

    ds = xr.open_mfdataset(
        files,
        combine="nested",
        concat_dim="time",
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )
    if ds.sizes["s_rho"] != vert_levels:
        raise ValueError(f"{his_glob} has {ds.sizes['s_rho']} levels, expected {vert_levels}")

    # Restart files repeat the boundary record: count each time once
    if "ocean_time" in ds:
        _, records = np.unique(ds["ocean_time"].values, return_index=True)
        records = np.sort(records)
    else:
        records = np.arange(ds.sizes["time"])
    records = records[::time_stride]

    sum_area_sigma = np.zeros(targets.size)
    sum_area = np.zeros(targets.size)
    for rec in records:
        snap = ds[["temp", "salt", "zeta"]].isel(time=rec).load()
        temp = snap["temp"].transpose("s_rho", "eta_rho", "xi_rho").values
        salt = snap["salt"].transpose("s_rho", "eta_rho", "xi_rho").values
        temp = temp.reshape(vert_levels, -1)[::-1, cols]  # top level first
        salt = salt.reshape(vert_levels, -1)[::-1, cols]
        zeta = snap["zeta"].transpose("eta_rho", "xi_rho").values.ravel()[cols]

        depth = roms_level_depths(zeta, h)
        sa = gsw.SR_from_SP(salt) * (1 + saar)  # equals gsw.SA_from_SP, with SAAR precomputed
        sigma0 = gsw.sigma0(sa, gsw.CT_from_pt(sa, temp))  # ROMS temp is potential temperature

        sigma_z = interp_to_depths(sigma0, depth, targets)
        valid = (targets[:, None] <= zeta + h) & np.isfinite(sigma_z)
        sum_area_sigma += np.where(valid, sigma_z, 0.0) @ area
        sum_area += valid @ area
    ds.close()
    return sum_area_sigma, sum_area, len(records)


def stevens_mld(depths, sigma):
    """Apply eq. S1 to a mean sigma0 profile on `depths` (surface first).

    Returns (mld, z_N, N^2 at the mid-depths). mld is NaN if the profile has no
    stable stratification.
    """
    n2 = G / RHO0 * np.diff(sigma) / np.diff(depths)
    i_n = np.nanargmax(n2)
    z_n = 0.5 * (depths[i_n] + depths[i_n + 1])
    sigma_n = 0.5 * (sigma[i_n] + sigma[i_n + 1])
    sigma_surf = sigma[0]
    if not sigma_n > sigma_surf:
        return np.nan, z_n, n2

    threshold = sigma_surf + 0.5 * (sigma_n - sigma_surf)
    k = np.argmax(sigma >= threshold)  # first level reaching the threshold (k >= 1)
    frac = (threshold - sigma[k - 1]) / (sigma[k] - sigma[k - 1])
    mld = depths[k - 1] + frac * (depths[k] - depths[k - 1])
    return mld, z_n, n2


# =========================
# Grid: vertical coordinate, ocean columns, cell areas, SAAR
# =========================
targets = np.asarray(mld_depth_levels, dtype=float)

roms_grid = Grid._from_file(model_grid_path)  # vertical coordinate read from the grid file
cs_r, sc_r = (np.asarray(v) for v in sigma_stretch(roms_grid.theta_s, roms_grid.theta_b, roms_grid.N, "r"))
hc = roms_grid.hc
vert_levels = roms_grid.N
print(f"Grid: N = {vert_levels}, theta_s = {roms_grid.theta_s}, theta_b = {roms_grid.theta_b}, hc = {hc}")

grid = xr.open_dataset(model_grid_path)
lon = ((grid["lon_rho"].values + 180) % 360) - 180
lat = grid["lat_rho"].values
keep = grid["mask_rho"].values == 1
if mld_region is not None:
    lon_min, lon_max, lat_min, lat_max = mld_region
    keep &= (lon >= lon_min) & (lon <= lon_max) & (lat >= lat_min) & (lat <= lat_max)
cols = np.flatnonzero(keep)
h = grid["h"].values.ravel()[cols]
area = (1 / (grid["pm"] * grid["pn"])).values.ravel()[cols]
saar = gsw.SAAR(0, lon.ravel()[cols], lat.ravel()[cols])
saar = saar[0] if isinstance(saar, tuple) else saar  # older gsw returns (SAAR, in_ocean)
grid.close()
print(f"Using {cols.size} ocean columns")

# =========================
# Time- and area-mean sigma0 for each member
# =========================
sums = {}
for ens, pattern in his_patterns.items():
    print(f"{ens}: averaging sigma0 over {pattern}")
    sums[ens] = accumulate_sigma_profile(pattern, cols, h, area, saar, targets, mld_time_stride)
    print(f"  {sums[ens][2]} history records")

# =========================
# Eq. S1 for each member and each season (pooled profiles)
# =========================
cases = {ens: [ens] for ens in his_patterns}
cases.update(seasons)

profiles, n2_profiles, z_n_values, mld_values = [], [], [], []
for case, members in cases.items():
    sum_area_sigma = sum(sums[m][0] for m in members)
    sum_area = sum(sums[m][1] for m in members)
    with np.errstate(invalid="ignore", divide="ignore"):
        sigma_mean = np.where(sum_area > 0, sum_area_sigma / sum_area, np.nan)
    mld, z_n, n2 = stevens_mld(targets, sigma_mean)
    profiles.append(sigma_mean)
    n2_profiles.append(n2)
    z_n_values.append(z_n)
    mld_values.append(mld)

ds_mld = xr.Dataset(
    data_vars={
        "mld": ("case", mld_values, {"units": "m", "long_name": "mixed-layer depth (Stevens 2010, eq. S1)"}),
        "z_N": ("case", z_n_values, {"units": "m", "long_name": "depth of maximum buoyancy frequency"}),
        "sigma0_mean": (
            ("case", "depth"),
            np.array(profiles),
            {"units": "kg m-3", "long_name": "time- and area-mean potential density anomaly"},
        ),
        "N2": (
            ("case", "depth_mid"),
            np.array(n2_profiles),
            {"units": "s-2", "long_name": "squared buoyancy frequency of the mean profile"},
        ),
    },
    coords={
        "case": list(cases),
        "depth": ("depth", targets, {"units": "m", "positive": "down", "long_name": "depth below the free surface"}),
        "depth_mid": ("depth_mid", 0.5 * (targets[1:] + targets[:-1]), {"units": "m", "positive": "down"}),
    },
    attrs={
        "description": "Mixed-layer depth following Gerke et al. (2016) and Stevens (2010), eq. S1",
        "seasons": "; ".join(f"{season}: {', '.join(members)}" for season, members in seasons.items()),
        "region": "all ocean points" if mld_region is None else f"lon/lat box {mld_region}",
        "time_stride": mld_time_stride,
    },
)

os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, "Iceland3_mld_stevens2010.nc")
ds_mld.to_netcdf(out_file)
print(f"\nSaved {out_file}\n")

print(f"{'case':<10}{'MLD (m)':>10}{'z_N (m)':>10}")
for case in cases:
    print(f"{case:<10}{float(ds_mld['mld'].sel(case=case)):>10.2f}{float(ds_mld['z_N'].sel(case=case)):>10.2f}")

# Seasonal MLD for every member: the upper-layer depths for the ALK-fraction cell in OAE_ensemble.ipynb
target_depths = {
    ens: round(float(ds_mld["mld"].sel(case=season)), 2) for season, members in seasons.items() for ens in members
}
print(f"\ntarget_depths = {target_depths}")

In [ ]:
import os

import matplotlib.pyplot as plt
import xarray as xr

# Check figure: mean sigma0 and N^2 profiles with z_N and the MLD for each season
ds_mld = xr.open_dataset(os.path.join(out_dir, "Iceland3_mld_stevens2010.nc"))

colors = {  # as in oae_analysis.ipynb
    "CDR1": "#1f77b4",
    "CDR2": "#2ca02c",
    "CDR3": "#ff7f0e",
    "CDR4": "#9467bd",
    "CDR5": "#8c564b",
    "CDR6": "#e377c2",
}

fig, axes = plt.subplots(2, len(seasons), figsize=(5.5 * len(seasons), 9), sharey=True, squeeze=False)
for col, (season, members) in enumerate(seasons.items()):
    ax_sigma, ax_n2 = axes[0, col], axes[1, col]
    for ens in members:
        ax_sigma.plot(ds_mld["sigma0_mean"].sel(case=ens), ds_mld["depth"], color=colors.get(ens), lw=1, label=ens)
        ax_n2.plot(ds_mld["N2"].sel(case=ens), ds_mld["depth_mid"], color=colors.get(ens), lw=1)
    ax_sigma.plot(ds_mld["sigma0_mean"].sel(case=season), ds_mld["depth"], color="k", lw=2, label=f"{season} mean")
    ax_n2.plot(ds_mld["N2"].sel(case=season), ds_mld["depth_mid"], color="k", lw=2)

    mld = float(ds_mld["mld"].sel(case=season))
    z_n = float(ds_mld["z_N"].sel(case=season))
    for ax in (ax_sigma, ax_n2):
        ax.axhline(mld, color="crimson", lw=1.5, label=f"MLD = {mld:.2f} m")
        ax.axhline(z_n, color="gray", ls="--", lw=1.5, label=f"$z_N$ = {z_n:.2f} m")
        ax.grid(True, ls="--", alpha=0.5)

    ax_sigma.set_title(rf"{season}: time- and area-mean $\sigma_0$", fontweight="bold")
    ax_n2.set_title(rf"{season}: $N^2$ of the mean profile", fontweight="bold")
    ax_sigma.set_xlabel(r"$\sigma_0$ (kg m$^{-3}$)")
    ax_n2.set_xlabel(r"$N^2$ (s$^{-2}$)")
    ax_sigma.legend(fontsize=8, loc="lower left")

for ax in axes[:, 0]:
    ax.set_ylabel("Depth below surface (m)")
axes[0, 0].set_ylim(30, 0)  # shared y axis: surface at the top

plt.tight_layout()
plt.show()
ds_mld.close()